# 24 m buffered evaluation: three seeds

Previously supplied runner with completed author-provided output. Uses saved split if available, otherwise explicitly reconstructs it. Retain generated protocol and exact membership lists.

Source: `gd_buffered24m_3seeds.py`.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""
GD-RGBMS-DB | FIXED 24 m BUFFERED SPLIT | SEEDS 42, 123, 2026
Paste this entire script into ONE Colab/Kaggle cell, or open the supplied notebook.

Paths and source verified against the COMPLETE graindiscoloration (2).ipynb:
  cell 67: original 24 m split generation (207/40/41)
  cell 70: completed 24 m auxiliary-gated model training
  cell 72: completed threshold rescan (0.20..0.95; seed-42 threshold 0.89)
Indices above are zero-based. The saved assignments CSV is reused if found.
Otherwise the exact original split algorithm is run ONCE and frozen for all seeds.
Same 70-epoch / patience-12 / AdamW / auxiliary-loss settings as the 24 m cell.
Normalization: 207 buffered TRAIN images only, shared by all three runs.
Checkpoint: best main-output validation Dice; final test: NO TTA.
Fine threshold search: 0.20..0.95, step 0.01, matching the completed rescan.
Threshold 0.89 is NOT fixed; each seed selects its threshold using validation.

Epoch-end checkpoints include optimizer, scheduler, scaler, and RNG state.
In Colab, outputs go to Google Drive by default so a new session can resume.
"""
from __future__ import annotations

import csv
import gc
import hashlib
import itertools
import json
import os
import random
import re
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ======================= NOTEBOOK PATHS — NO PLACEHOLDERS ====================
BASE_DIR = ("/kaggle/input/datasets/hemal7/gd-rgbms-db"
            if Path("/kaggle/input/datasets/hemal7/gd-rgbms-db").exists()
            else "/content/GD-RGBMS-DB" if Path("/content/GD-RGBMS-DB").exists() else None)
SPATIAL_SPLIT_PATH = None  # automatic original CSV / ZIP discovery below
SPLIT_DIR = None           # optional override, normally leave unchanged
SPLIT_ZIP = None           # optional override, normally leave unchanged
OUTPUT_DIR = None          # actual output folder set automatically below
MOUNT_GOOGLE_DRIVE = True

RUN_SEEDS = [42, 123, 2026]
RESUME = True
SAVE_TEST_PREDICTIONS = True
CACHE_TRAIN_AND_VALIDATION = True

BUFFER_M = 24.0
EXPECTED_COUNTS = {"train":207, "val":40, "test":41}
TARGET_SIZE = (896, 672)  # width, height
BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
MAX_EPOCHS = 70
PATIENCE = 12
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
BRANCH_BASE = 28
PREFERRED_GN_GROUPS = 8
MODALITY_DROPOUT_PROBABILITY = 0.0
RGB_CHANNELS = 3
SPECTRAL_CHANNELS = 3
TOTAL_INPUT_CHANNELS = 6
INPUT_CHANNEL_ORDER = ("R", "G", "B", "Green", "Red-Edge", "NIR")
MASK_POSITIVE_RGB = np.array([240,128,128], dtype=np.int16)
AUX_RGB_INITIAL_WEIGHT, AUX_SPECTRAL_INITIAL_WEIGHT = 0.15, 0.25
AUX_RGB_MIDDLE_WEIGHT, AUX_SPECTRAL_MIDDLE_WEIGHT = 0.10, 0.15
AUX_RGB_FINAL_WEIGHT, AUX_SPECTRAL_FINAL_WEIGHT = 0.05, 0.08
AUX_MIDDLE_PHASE_START, AUX_FINAL_PHASE_START = 0.40, 0.70
TRAIN_THRESHOLDS = np.round(np.arange(0.20, 0.851, 0.05),2).tolist()
FINAL_THRESHOLDS = np.round(np.arange(0.20, 0.951, 0.01),2).tolist()
METRIC_NAMES = ("dice","iou","precision","recall","accuracy")
BILINEAR = Image.Resampling.BILINEAR
NEAREST = Image.Resampling.NEAREST

def read_list(path):
    return [s.strip() for s in Path(path).read_text(encoding="utf-8-sig").splitlines() if s.strip()]

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def save_json(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)

def save_csv(path, rows):
    if not rows:
        return
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with tmp.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    os.replace(tmp, path)

def atomic_torch_save(value, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(value, tmp)
    os.replace(tmp, path)

def load_checkpoint(path):
    # Load only checkpoints created by this runner in your own output folder.
    return torch.load(path, map_location="cpu", weights_only=False)

def valid_dataset(path):
    return all((path/p).is_dir() for p in ("RGB","MS","Mark"))

def discover_dataset(roots):
    if BASE_DIR is not None:
        path = Path(BASE_DIR)
        if not valid_dataset(path):
            raise FileNotFoundError(f"BASE_DIR must contain RGB/, MS/, Mark/: {path}")
        return path
    for path in (Path('/content/GD-RGBMS-DB'), Path('/kaggle/input/datasets/hemal7/gd-rgbms-db')):
        if valid_dataset(path):
            return path
    found = set()
    for root in roots:
        if root.exists():
            found.update(p.parent.resolve() for p in root.rglob('RGB') if valid_dataset(p.parent))
    if not found:
        raise FileNotFoundError('GD dataset not found. Set BASE_DIR to the folder containing RGB/, MS/, Mark/.')
    preferred = [p for p in found if 'gd-rgbms' in str(p).lower()]
    choices = preferred or list(found)
    if len(choices) != 1:
        raise RuntimeError('Multiple datasets found. Set BASE_DIR explicitly: ' + str(sorted(map(str,choices))))
    return choices[0]

def has_split_lists(path):
    return all((path/f'{s}_list.txt').is_file() for s in EXPECTED_COUNTS)

def has_expected_counts(path):
    if not has_split_lists(path):
        return False
    try:
        return all(len(read_list(path/f'{s}_list.txt')) == n for s,n in EXPECTED_COUNTS.items())
    except (OSError, UnicodeError):
        return False

def split_content_id(path):
    text = json.dumps({s:[sample_stem(n) for n in read_list(path/f'{s}_list.txt')]
                       for s in EXPECTED_COUNTS}, sort_keys=True)
    return hashlib.sha256(text.encode()).hexdigest()

def safe_extract_split_zip(path, extraction_root):
    destination = extraction_root/hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        for item in archive.infolist():
            candidate = (destination/item.filename).resolve()
            if not candidate.is_relative_to(destination.resolve()):
                raise ValueError('Unsafe path in split ZIP.')
        archive.extractall(destination)
    return destination

def discover_split(roots, work_dir):
    if SPLIT_DIR is not None:
        path = Path(SPLIT_DIR)
        if not has_expected_counts(path):
            raise RuntimeError(f'SPLIT_DIR must contain the ORIGINAL 207/40/41 lists: {path}')
        return path
    if SPLIT_ZIP is not None:
        roots = [safe_extract_split_zip(Path(SPLIT_ZIP), work_dir/'buffered24m_split_inputs')]
    candidates = set()
    for root in roots:
        if not root.exists():
            continue
        candidates.update(p.parent.resolve() for p in root.rglob('train_list.txt')
                          if has_expected_counts(p.parent))
    if not candidates and SPLIT_ZIP is None:
        # Only inspect likely spatial-split archives, never invent a new split.
        zips = set()
        for root in roots:
            if root.exists():
                zips.update(root.rglob('*spatial*split*.zip'))
                zips.update(root.rglob('*24m*.zip'))
        for path in sorted(zips):
            try:
                extracted = safe_extract_split_zip(path, work_dir/'buffered24m_split_inputs')
            except zipfile.BadZipFile:
                continue
            candidates.update(p.parent.resolve() for p in extracted.rglob('train_list.txt')
                              if has_expected_counts(p.parent))
    if not candidates:
        raise FileNotFoundError(
            'Original 24 m split lists were not found. Set SPLIT_DIR or SPLIT_ZIP to your saved '
            '207 train / 40 validation / 41 test split. Counts alone cannot reproduce the original split.')
    by_id = {}
    for path in sorted(candidates):
        by_id.setdefault(split_content_id(path), path)
    if len(by_id) != 1:
        raise RuntimeError('Several different 207/40/41 splits found. Set SPLIT_DIR explicitly: '
                           + str(list(map(str,by_id.values()))))
    return next(iter(by_id.values()))

def prepare_assignment_lists(dataset,roots,work_dir):
    """Reuse the notebook's real CSV or ZIP; otherwise recreate its exact split."""
    candidates = []
    if SPATIAL_SPLIT_PATH is not None:
        supplied = Path(SPATIAL_SPLIT_PATH)
        if supplied.suffix.lower()=='.zip':
            extracted = safe_extract_split_zip(supplied,work_dir/'spatial_24m_split_input')
            candidates = list(extracted.rglob('spatial_24m_split_assignments.csv'))
        elif supplied.is_file():
            candidates = [supplied]
        else:
            raise FileNotFoundError(f'Spatial split file is missing: {supplied}')
    else:
        preferred = [
            Path('/kaggle/working/spatial_24m_split/spatial_24m_split_assignments.csv'),
            Path('/kaggle/input/datasets/hemal7/gd-spatial24m-gatedfusion-seed42-res/spatial_24m_split_assignments.csv'),
            Path('/content/spatial_24m_split/spatial_24m_split_assignments.csv')]
        candidates = [p for p in preferred if p.is_file()]
        for root in roots:
            if root.exists():
                candidates.extend(root.rglob('spatial_24m_split_assignments.csv'))
        if not candidates:
            for root in roots:
                if not root.exists():
                    continue
                for path in root.rglob('spatial_24m_leakage_aware_split.zip'):
                    extracted = safe_extract_split_zip(path,work_dir/'spatial_24m_split_input')
                    candidates.extend(extracted.rglob('spatial_24m_split_assignments.csv'))
    if not candidates:
        if SPATIAL_SPLIT_PATH is not None:
            raise FileNotFoundError('The supplied split ZIP does not contain the original assignments CSV.')
        print('Original saved CSV absent: recreating the notebook 24 m split ONCE.')
        candidates = [regenerate_original_24m_split(dataset,work_dir)]
    identities = {}
    for path in dict.fromkeys(candidates):
        with path.open(newline='',encoding='utf-8-sig') as f:
            rows = list(csv.DictReader(f))
        if not rows or not {'sample_name','latitude','longitude','split'}.issubset(rows[0]):
            raise ValueError(f'Invalid assignments CSV: {path}')
        counts = {s:sum(r['split'].strip()==s for r in rows)
                  for s in ('train','validation','test','excluded_buffer')}
        if counts != {'train':207,'validation':40,'test':41,'excluded_buffer':112}:
            raise ValueError(f'Unexpected split counts in {path}: {counts}')
        if len(rows)!=400 or len({sample_stem(r['sample_name']) for r in rows})!=400:
            raise ValueError('Assignments CSV must contain all 400 distinct images.')
        identity = hashlib.sha256(json.dumps([(sample_stem(r['sample_name']),r['split'].strip())
                                             for r in rows]).encode()).hexdigest()
        identities.setdefault(identity,(path,rows))
    if len(identities)!=1:
        raise RuntimeError('Conflicting original 24 m assignments found; set SPATIAL_SPLIT_PATH explicitly.')
    identity,(path,rows) = next(iter(identities.items()))
    destination = work_dir/'spatial_24m_fixed_runner_lists'/identity[:16]
    destination.mkdir(parents=True,exist_ok=True)
    for output_label,csv_label in [('train','train'),('val','validation'),('test','test'),
                                  ('excluded_buffer','excluded_buffer')]:
        names = [r['sample_name'].strip() for r in rows if r['split'].strip()==csv_label]
        (destination/f'{output_label}_list.txt').write_text('\n'.join(names)+'\n')
    shutil.copy2(path,destination/'spatial_24m_split_assignments.csv')
    print('Original 24 m assignments:',path)
    return destination

def sample_stem(name):
    path = Path(name.replace('\\','/'))
    return path.stem if path.suffix.lower() in ('.jpg','.jpeg','.png') else path.name

def normalize_names(names, canonical):
    result = []
    for name in names:
        stem = sample_stem(name)
        if stem not in canonical:
            raise ValueError(f'Split sample is absent from the RGB dataset: {name}')
        result.append(canonical[stem])
    if len(result) != len(set(result)):
        raise ValueError('Duplicate image identifiers in a split list.')
    return result

def validate_split(split_dir, canonical):
    splits = {s:normalize_names(read_list(split_dir/f'{s}_list.txt'),canonical)
              for s in EXPECTED_COUNTS}
    for s,n in EXPECTED_COUNTS.items():
        if len(splits[s]) != n:
            raise ValueError(f'Wrong {s} size: {len(splits[s])}; expected {n}.')
    for a,b in itertools.combinations(splits,2):
        if set(splits[a]) & set(splits[b]):
            raise ValueError(f'Image overlap between {a} and {b}.')
    if len(canonical) != 400:
        raise ValueError(f'Expected 400 unique RGB images; found {len(canonical)}.')
    active = set().union(*(set(v) for v in splits.values()))
    excluded = sorted(set(canonical.values())-active)
    for filename in ('excluded_buffer_list.txt','excluded_buffer.txt','excluded_list.txt'):
        path = split_dir/filename
        if path.exists() and set(normalize_names(read_list(path),canonical)) != set(excluded):
            raise ValueError('Saved excluded list does not match the complement of the active split.')
    summary = split_dir/'split_summary.json'
    if summary.exists():
        info = json.loads(summary.read_text())
        for key in ('safety_buffer_m','spatial_buffer_m','buffer_m'):
            if key in info and not np.isclose(float(info[key]), BUFFER_M):
                raise ValueError(f'Saved split metadata says {info[key]} m, expected 24 m.')
    return splits, excluded

GPS_PATTERN = re.compile(r'lat_(-?\d+(?:\.\d+)?)_lon_(-?\d+(?:\.\d+)?)', re.I)

def gps_coordinates(names):
    coordinates = []
    for name in names:
        m = GPS_PATTERN.search(Path(name).stem)
        if m is None:
            raise ValueError(f'Cannot audit 24 m separation: GPS is missing in {name}.')
        coordinates.append([float(m.group(1)),float(m.group(2))])
    return np.radians(np.asarray(coordinates,dtype=np.float64))

def haversine_matrix(a,b):
    dlat = b[None,:,0]-a[:,None,0]
    dlon = b[None,:,1]-a[:,None,1]
    h = np.sin(dlat/2)**2 + np.cos(a[:,None,0])*np.cos(b[None,:,0])*np.sin(dlon/2)**2
    return 2*6371000.0*np.arcsin(np.sqrt(np.clip(h,0,1)))

def audit_buffer(splits):
    coordinates = {s:gps_coordinates(v) for s,v in splits.items()}
    rows = []
    for a,b in itertools.combinations(splits,2):
        distances = haversine_matrix(coordinates[a], coordinates[b])
        minimum = float(distances.min())
        violations = int(np.count_nonzero(distances < BUFFER_M-1e-6))
        if violations:
            raise ValueError(f'24 m audit FAILED: {a}/{b} minimum {minimum:.6f} m; '
                             f'{violations} pairs below 24 m. Training was not started.')
        rows.append({'comparison':f'{a}_vs_{b}', 'minimum_camera_centre_distance_m':minimum,
                     'pairs_below_24m':violations})
    return rows

def initialize_inputs():
    if MOUNT_GOOGLE_DRIVE and Path('/content').exists() and not Path('/kaggle/working').exists():
        try:
            from google.colab import drive
        except ImportError:
            pass
        else:
            if not Path('/content/drive/MyDrive').exists():
                drive.mount('/content/drive')
    work = Path('/kaggle/working') if Path('/kaggle/working').exists() else (
        Path('/content') if Path('/content').exists() else Path.cwd())
    roots = [Path('/kaggle/input'),work,Path('/content/drive/MyDrive')]
    dataset = discover_dataset(roots)
    split = (discover_split(roots,work) if SPLIT_DIR is not None or SPLIT_ZIP is not None
             else prepare_assignment_lists(dataset,roots,work))
    rgb_files = [p for p in (dataset/'RGB').iterdir()
                 if p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.png')]
    canonical = {p.stem:p.name for p in rgb_files}
    if len(canonical) != len(rgb_files):
        raise ValueError('Ambiguous RGB files with the same stem.')
    splits, excluded = validate_split(split,canonical)
    audit = audit_buffer(splits)
    # Confirm all active paired MS and mask files before any expensive training.
    for name in set().union(*(set(v) for v in splits.values())):
        stem = Path(name).stem
        find_path(dataset/'MS',stem,['.PNG','.png','.JPG','.jpg','.JPEG','.jpeg'])
        find_path(dataset/'Mark',stem,['.png','.PNG','.jpg','.JPG'])
    if OUTPUT_DIR is not None:
        out = Path(OUTPUT_DIR)
    elif work == Path('/content') and Path('/content/drive/MyDrive').exists():
        out = Path('/content/drive/MyDrive/gd_spatial24m_gatedfusion_3seeds')
    else:
        out = work/'gd_spatial24m_gatedfusion_3seeds'
    out.mkdir(parents=True,exist_ok=True)
    return dataset,split,splits,excluded,audit,out


# === Original 24 m split generation from the complete notebook ===

def regenerate_original_24m_split(dataset, work_dir):
    """Original notebook split-generation cell; run ONCE before seed training."""
    import pandas as pd
    from scipy.spatial.distance import cdist
    from scipy.sparse.csgraph import connected_components
    from scipy.sparse import csr_matrix
    RGB_DIR = dataset / 'RGB'
    OUTPUT_DIR = work_dir / 'spatial_24m_split'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    TARGET_VAL = TARGET_TEST = 40
    VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
    LAT_LON_PATTERN = GPS_PATTERN
    records = []
    
    for image_path in sorted(RGB_DIR.rglob("*")):
        if not image_path.is_file():
            continue
    
        if image_path.suffix.lower() not in VALID_EXTENSIONS:
            continue
    
        match = LAT_LON_PATTERN.search(image_path.stem)
    
        if match is None:
            continue
    
        records.append({
            "sample_name": image_path.stem,
            "latitude": float(match.group(1)),
            "longitude": float(match.group(2)),
            "rgb_path": str(image_path),
        })
    
    df = pd.DataFrame(records)
    
    # Avoid accidental duplicate files
    df = (
        df.drop_duplicates(subset="sample_name")
          .sort_values("sample_name")
          .reset_index(drop=True)
    )
    
    print("\nUnique samples:", len(df))
    
    if len(df) != 400:
        print(
            f"WARNING: Expected 400 samples, but found {len(df)}. "
            "The code will continue using the detected samples."
        )
    
    
    # ============================================================
    # 3. Convert latitude/longitude to local metre coordinates
    # ============================================================
    
    EARTH_RADIUS_M = 6_371_000.0
    
    mean_latitude_rad = np.deg2rad(df["latitude"].mean())
    mean_longitude = df["longitude"].mean()
    mean_latitude = df["latitude"].mean()
    
    df["x_m"] = (
        np.deg2rad(df["longitude"] - mean_longitude)
        * EARTH_RADIUS_M
        * np.cos(mean_latitude_rad)
    )
    
    df["y_m"] = (
        np.deg2rad(df["latitude"] - mean_latitude)
        * EARTH_RADIUS_M
    )
    
    coordinates = df[["x_m", "y_m"]].to_numpy()
    
    
    # ============================================================
    # 4. Identify independent spatial fields/components
    # ============================================================
    
    distance_matrix = cdist(coordinates, coordinates)
    
    adjacency = (
        (distance_matrix <= BUFFER_M)
        & (distance_matrix > 0)
    )
    
    number_of_components, component_labels = connected_components(
        csr_matrix(adjacency.astype(np.uint8)),
        directed=False,
        return_labels=True
    )
    
    df["spatial_component_24m"] = component_labels
    
    component_sizes = (
        df.groupby("spatial_component_24m")
          .size()
          .sort_values()
    )
    
    print("\n24 m spatial components:")
    print(component_sizes.to_string())
    
    if number_of_components != 2:
        raise ValueError(
            f"Expected 2 major spatial components, but found "
            f"{number_of_components}. Please send the printed output "
            "before proceeding."
        )
    
    
    # ============================================================
    # 5. Find the best buffered end-region within each field
    # ============================================================
    
    def find_best_buffered_end_split(
        component_df,
        target_holdout,
        buffer_m=24.0,
        target_tolerance=3
    ):
        """
        Selects one end of a spatial field as validation/test.
        The opposite side becomes training.
        Images inside the buffer between them are excluded.
        """
    
        component_points = component_df[["x_m", "y_m"]].to_numpy()
        n = len(component_df)
    
        best_result = None
    
        # Search many spatial directions
        angles = np.linspace(0.0, 179.75, 1440)
    
        minimum_k = max(15, target_holdout - target_tolerance)
        maximum_k = min(n - 15, target_holdout + target_tolerance)
    
        for angle_degree in angles:
            angle_radian = np.deg2rad(angle_degree)
    
            direction = np.array([
                np.cos(angle_radian),
                np.sin(angle_radian)
            ])
    
            projected_position = component_points @ direction
            sorted_indices = np.argsort(projected_position)
    
            for side in ["low", "high"]:
                for k in range(minimum_k, maximum_k + 1):
    
                    if side == "low":
                        holdout_local = sorted_indices[:k]
                        holdout_boundary = projected_position[
                            holdout_local
                        ].max()
    
                        train_local = np.where(
                            projected_position
                            > holdout_boundary + buffer_m
                        )[0]
    
                        excluded_local = np.where(
                            (projected_position > holdout_boundary)
                            & (
                                projected_position
                                <= holdout_boundary + buffer_m
                            )
                        )[0]
    
                    else:
                        holdout_local = sorted_indices[-k:]
                        holdout_boundary = projected_position[
                            holdout_local
                        ].min()
    
                        train_local = np.where(
                            projected_position
                            < holdout_boundary - buffer_m
                        )[0]
    
                        excluded_local = np.where(
                            (
                                projected_position
                                >= holdout_boundary - buffer_m
                            )
                            & (
                                projected_position
                                < holdout_boundary
                            )
                        )[0]
    
                    if len(train_local) == 0:
                        continue
    
                    minimum_distance = cdist(
                        component_points[holdout_local],
                        component_points[train_local]
                    ).min()
    
                    if minimum_distance < buffer_m:
                        continue
    
                    # Priority:
                    # 1. Retain the maximum number of images
                    # 2. Stay close to requested holdout size
                    # 3. Prefer slightly larger separation
                    score = (
                        len(holdout_local) + len(train_local),
                        -abs(len(holdout_local) - target_holdout),
                        minimum_distance
                    )
    
                    if (
                        best_result is None
                        or score > best_result["score"]
                    ):
                        best_result = {
                            "score": score,
                            "angle_degree": angle_degree,
                            "side": side,
                            "holdout_local": holdout_local,
                            "train_local": train_local,
                            "excluded_local": excluded_local,
                            "minimum_distance": minimum_distance,
                        }
    
        if best_result is None:
            raise RuntimeError(
                "No valid buffered split was found for this component."
            )
    
        component_global_indices = component_df.index.to_numpy()
    
        best_result["holdout_global"] = component_global_indices[
            best_result["holdout_local"]
        ]
    
        best_result["train_global"] = component_global_indices[
            best_result["train_local"]
        ]
    
        best_result["excluded_global"] = component_global_indices[
            best_result["excluded_local"]
        ]
    
        return best_result
    
    
    # Smaller field supplies validation; larger field supplies test.
    # Both fields still contribute training images.
    sorted_components = component_sizes.index.tolist()
    
    validation_component = sorted_components[0]
    test_component = sorted_components[1]
    
    validation_result = find_best_buffered_end_split(
        df[df["spatial_component_24m"] == validation_component],
        target_holdout=TARGET_VAL,
        buffer_m=BUFFER_M
    )
    
    test_result = find_best_buffered_end_split(
        df[df["spatial_component_24m"] == test_component],
        target_holdout=TARGET_TEST,
        buffer_m=BUFFER_M
    )
    
    
    # ============================================================
    # 6. Assign final split labels
    # ============================================================
    
    df["split"] = "excluded_buffer"
    
    df.loc[validation_result["train_global"], "split"] = "train"
    df.loc[test_result["train_global"], "split"] = "train"
    
    df.loc[
        validation_result["holdout_global"],
        "split"
    ] = "validation"
    
    df.loc[
        test_result["holdout_global"],
        "split"
    ] = "test"
    
    
    # ============================================================
    # 7. Verify that no cross-set pair is within 24 m
    # ============================================================
    
    def cross_split_statistics(split_a, split_b):
        indices_a = df.index[df["split"] == split_a].to_numpy()
        indices_b = df.index[df["split"] == split_b].to_numpy()
    
        distances = distance_matrix[np.ix_(indices_a, indices_b)]
    
        return {
            "split_a": split_a,
            "split_b": split_b,
            "minimum_distance_m": float(distances.min()),
            "pairs_at_or_below_24m": int(
                np.sum(distances <= BUFFER_M)
            ),
        }
    
    
    verification = pd.DataFrame([
        cross_split_statistics("train", "validation"),
        cross_split_statistics("train", "test"),
        cross_split_statistics("validation", "test"),
    ])
    
    print("\nCross-split verification:")
    print(verification.to_string(index=False))
    
    if not (verification["pairs_at_or_below_24m"] == 0).all():
        raise AssertionError(
            "Verification failed: at least one cross-set pair "
            "is within 24 m."
        )
    
    print("\n✅ VERIFIED: No cross-set pair is within 24 m.")
    
    
    # ============================================================
    # 8. Save assignment and split files
    # ============================================================
    
    split_order = [
        "train",
        "validation",
        "test",
        "excluded_buffer"
    ]
    
    summary = (
        df["split"]
        .value_counts()
        .reindex(split_order, fill_value=0)
        .rename_axis("split")
        .reset_index(name="number_of_images")
    )
    
    retained = int(
        summary.loc[
            summary["split"] != "excluded_buffer",
            "number_of_images"
        ].sum()
    )
    
    print("\nFinal split summary:")
    print(summary.to_string(index=False))
    print("\nTotal retained:", retained)
    print("Total excluded:", int((df["split"] == "excluded_buffer").sum()))
    
    assignment_columns = [
        "sample_name",
        "latitude",
        "longitude",
        "x_m",
        "y_m",
        "spatial_component_24m",
        "split",
        "rgb_path",
    ]
    
    df[assignment_columns].to_csv(
        OUTPUT_DIR / "spatial_24m_split_assignments.csv",
        index=False
    )
    
    summary.to_csv(
        OUTPUT_DIR / "spatial_24m_split_summary.csv",
        index=False
    )
    
    verification.to_csv(
        OUTPUT_DIR / "spatial_24m_split_verification.csv",
        index=False
    )
    
    for split_name in split_order:
        split_df = df[df["split"] == split_name][assignment_columns]
    
        split_df.to_csv(
            OUTPUT_DIR / f"{split_name}_spatial_24m.csv",
            index=False
        )
    
    
    return OUTPUT_DIR / 'spatial_24m_split_assignments.csv'


# === Actual 24 m notebook image loader, architecture, loss, and metrics ===

def find_path(directory, stem, extensions):
    for extension in extensions:
        path = directory / f"{stem}{extension}"
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {stem} in {directory}")

def binary_mask(image):
    array = np.asarray(image.convert("RGB"), dtype=np.int16)
    return (
        np.abs(array - MASK_POSITIVE_RGB).sum(axis=2) < 80
    ).astype(np.float32)

def load_raw(name):
    stem = sample_stem(name)
    rgb_path = find_path(rgb_dir, stem, [".JPG", ".jpg", ".JPEG", ".jpeg"])
    ms_path = find_path(ms_dir, stem, [".PNG", ".png"])
    mark_path = find_path(mark_dir, stem, [".png", ".PNG"])

    with Image.open(rgb_path) as item:
        rgb_image = item.convert("RGB")
    with Image.open(ms_path) as item:
        ms_image = item.convert("RGB")
    with Image.open(mark_path) as item:
        mark_image = item.convert("RGB")

    rgb_image = rgb_image.resize(TARGET_SIZE, BILINEAR)
    ms_image = ms_image.resize(TARGET_SIZE, BILINEAR)
    mark_image = mark_image.resize(TARGET_SIZE, NEAREST)

    rgb = np.asarray(rgb_image, dtype=np.float32).copy() / 255.0
    ms = np.asarray(ms_image, dtype=np.float32).copy() / 255.0
    mask = binary_mask(mark_image)
    image = np.concatenate([rgb, ms], axis=2)
    if image.shape[2] != TOTAL_INPUT_CHANNELS:
        raise RuntimeError(
            f"Expected {TOTAL_INPUT_CHANNELS} input channels but received "
            f"{image.shape[2]} for {stem}."
        )
    return image.astype(np.float32), mask, stem

def group_norm(channels):
    groups = min(PREFERRED_GN_GROUPS, channels)
    while channels % groups:
        groups -= 1
    return nn.GroupNorm(groups, channels, eps=1e-5, affine=True)

class SCSE(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.channel = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )
        self.spatial = nn.Sequential(nn.Conv2d(channels, 1, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.channel(x) + x * self.spatial(x)

class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(dropout) if dropout else nn.Identity(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
        )
        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                group_norm(out_channels),
            )
        )
        self.attention = SCSE(out_channels)

    def forward(self, x):
        return self.attention(
            F.silu(self.block(x) + self.skip(x), inplace=True)
        )

class SoftGatedFusion(nn.Module):
    """Two learned weights sum to one at every spatial location."""

    def __init__(self, channels):
        super().__init__()
        hidden = max(channels // 8, 8)
        self.gate = nn.Sequential(
            nn.Conv2d(channels * 2, hidden, 1, bias=False),
            group_norm(hidden),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, 2, 1),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            group_norm(channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, rgb_feature, spectral_feature):
        weights = torch.softmax(
            self.gate(torch.cat([rgb_feature, spectral_feature], dim=1)),
            dim=1,
        )
        fused = (
            weights[:, 0:1] * rgb_feature
            + weights[:, 1:2] * spectral_feature
        )
        return self.refine(fused), weights

class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.conv = ResBlock(
            in_channels + skip_channels, out_channels, dropout
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x, size=skip.shape[-2:], mode="bilinear", align_corners=False
        )
        return self.conv(torch.cat([x, skip], dim=1))

class AuxiliarySupervisedGatedUNetNoNDRE(nn.Module):
    def __init__(self, base=28):
        super().__init__()
        channels = [base, base * 2, base * 4, base * 8]
        c1, c2, c3, c4 = channels
        self.pool = nn.MaxPool2d(2)

        self.rgb1 = ResBlock(RGB_CHANNELS, c1)
        self.spec1 = ResBlock(SPECTRAL_CHANNELS, c1)
        self.rgb2, self.spec2 = ResBlock(c1, c2), ResBlock(c1, c2)
        self.rgb3, self.spec3 = ResBlock(c2, c3, 0.05), ResBlock(c2, c3, 0.05)
        self.rgb4, self.spec4 = ResBlock(c3, c4, 0.10), ResBlock(c3, c4, 0.10)

        self.fuse1 = SoftGatedFusion(c1)
        self.fuse2 = SoftGatedFusion(c2)
        self.fuse3 = SoftGatedFusion(c3)
        self.fuse4 = SoftGatedFusion(c4)

        # Training-only heads. Their outputs are requested explicitly by
        # return_aux=True; ordinary inference does not execute them.
        self.rgb_auxiliary_head = nn.Conv2d(c4, 1, kernel_size=1)
        self.spectral_auxiliary_head = nn.Conv2d(c4, 1, kernel_size=1)

        self.center = ResBlock(c4, c4 * 2, 0.10)
        self.up4 = UpBlock(c4 * 2, c4, c4, 0.10)
        self.up3 = UpBlock(c4, c3, c3, 0.05)
        self.up2 = UpBlock(c3, c2, c2)
        self.up1 = UpBlock(c2, c1, c1)
        self.output = nn.Conv2d(c1, 1, 1)

    def modality_dropout(self, rgb, spectral):
        if not self.training or MODALITY_DROPOUT_PROBABILITY <= 0:
            return rgb, spectral
        draw = random.random()
        if draw < MODALITY_DROPOUT_PROBABILITY / 2:
            rgb = torch.zeros_like(rgb)
        elif draw < MODALITY_DROPOUT_PROBABILITY:
            spectral = torch.zeros_like(spectral)
        return rgb, spectral

    def forward(self, x, return_gates=False, return_aux=False):
        if x.ndim != 4 or x.shape[1] != TOTAL_INPUT_CHANNELS:
            raise ValueError(
                "Expected input shaped [B, 6, H, W] in the order "
                f"{INPUT_CHANNEL_ORDER}; received {tuple(x.shape)}."
            )
        input_size = x.shape[-2:]
        rgb, spectral = self.modality_dropout(
            x[:, :RGB_CHANNELS], x[:, RGB_CHANNELS:]
        )

        r1, s1 = self.rgb1(rgb), self.spec1(spectral)
        f1, g1 = self.fuse1(r1, s1)

        r2 = self.rgb2(self.pool(r1 + f1))
        s2 = self.spec2(self.pool(s1 + f1))
        f2, g2 = self.fuse2(r2, s2)

        r3 = self.rgb3(self.pool(r2 + f2))
        s3 = self.spec3(self.pool(s2 + f2))
        f3, g3 = self.fuse3(r3, s3)

        r4 = self.rgb4(self.pool(r3 + f3))
        s4 = self.spec4(self.pool(s3 + f3))
        f4, g4 = self.fuse4(r4, s4)

        center = self.center(self.pool(f4))
        d4 = self.up4(center, f4)
        d3 = self.up3(d4, f3)
        d2 = self.up2(d3, f2)
        d1 = self.up1(d2, f1)
        logits = self.output(d1)

        gates = [g1, g2, g3, g4]
        if return_aux:
            rgb_auxiliary_logits = F.interpolate(
                self.rgb_auxiliary_head(r4),
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )
            spectral_auxiliary_logits = F.interpolate(
                self.spectral_auxiliary_head(s4),
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )
            if return_gates:
                return (
                    logits,
                    gates,
                    rgb_auxiliary_logits,
                    spectral_auxiliary_logits,
                )
            return logits, rgb_auxiliary_logits, spectral_auxiliary_logits

        return (logits, gates) if return_gates else logits

class ComboLoss(nn.Module):
    def forward(self, logits, targets):
        bce_map = F.binary_cross_entropy_with_logits(
            logits, targets.float(), reduction="none"
        )
        bce = bce_map.mean()
        probabilities = torch.sigmoid(logits)
        pt = (
            probabilities * targets
            + (1 - probabilities) * (1 - targets)
        )
        focal = (bce_map * ((1 - pt) ** 2.0)).mean()
        dims = tuple(range(1, logits.ndim))
        intersection = (probabilities * targets).sum(dims)
        denominator = probabilities.sum(dims) + targets.sum(dims)
        dice = 1 - (
            (2 * intersection + 1e-6) / (denominator + 1e-6)
        ).mean()
        return 0.35 * bce + 0.25 * focal + 0.40 * dice

def confusion(probabilities, masks, threshold):
    prediction, target = probabilities >= threshold, masks >= 0.5
    return [
        int((prediction & target).sum().item()),
        int((prediction & ~target).sum().item()),
        int((~prediction & target).sum().item()),
        int((~prediction & ~target).sum().item()),
    ]

def metrics(values):
    tp, fp, fn, tn = values
    e = 1e-7
    return {
        "dice": (2 * tp + e) / (2 * tp + fp + fn + e),
        "iou": (tp + e) / (tp + fp + fn + e),
        "precision": (tp + e) / (tp + fp + e),
        "recall": (tp + e) / (tp + fn + e),
        "accuracy": (tp + tn + e) / (tp + fp + fn + tn + e),
    }

def auxiliary_loss_weights(epoch):
    """Use an interpretable three-phase auxiliary-loss schedule."""
    progress = epoch / max(MAX_EPOCHS, 1)
    if progress <= AUX_MIDDLE_PHASE_START:
        return AUX_RGB_INITIAL_WEIGHT, AUX_SPECTRAL_INITIAL_WEIGHT
    if progress <= AUX_FINAL_PHASE_START:
        return AUX_RGB_MIDDLE_WEIGHT, AUX_SPECTRAL_MIDDLE_WEIGHT
    return AUX_RGB_FINAL_WEIGHT, AUX_SPECTRAL_FINAL_WEIGHT

def build_model():
    model = AuxiliarySupervisedGatedUNetNoNDRE(BRANCH_BASE).to(device)
    parameter_count = sum((p.numel() for p in model.parameters()))
    auxiliary_parameter_count = sum((p.numel() for n, p in model.named_parameters() if 'auxiliary_head' in n))
    deployment_parameter_count = parameter_count - auxiliary_parameter_count
    if parameter_count != 7950483 or deployment_parameter_count != 7950033:
        raise RuntimeError('Architecture mismatch: parameter count differs from the locked six-channel proposed model.')
    return (model, parameter_count, auxiliary_parameter_count, deployment_parameter_count)

# ======================= FIXED DATA / NORMALIZATION ==========================

def prepare_data(train_names,val_names):
    cache = {}
    sums = np.zeros(6,dtype=np.float64)
    squares = np.zeros(6,dtype=np.float64)
    pixels = 0
    training = set(train_names)
    for name in tqdm(train_names+val_names,desc='Cache images / buffered TRAIN normalization'):
        image,mask,stem = load_raw(name)
        if not np.isfinite(image).all():
            raise ValueError(f'Invalid image values: {name}')
        if name in training:
            sums += image.sum(axis=(0,1),dtype=np.float64)
            squares += np.square(image,dtype=np.float32).sum(axis=(0,1),dtype=np.float64)
            pixels += image.shape[0]*image.shape[1]
        if CACHE_TRAIN_AND_VALIDATION:
            cache[name] = (image.astype(np.float16),mask.astype(np.uint8),stem)
    mean = sums/pixels
    std = np.sqrt(np.maximum(squares/pixels-mean**2,1e-12))
    return cache,mean.astype(np.float32),std.astype(np.float32)

class BufferedDataset(Dataset):
    def __init__(self,names,augment=False):
        self.names,self.augment = list(names),augment

    def __len__(self):
        return len(self.names)

    def __getitem__(self,index):
        name = self.names[index]
        if name in RAM_CACHE:
            image,mask,stem = RAM_CACHE[name]
            image,mask = image.astype(np.float32),mask.astype(np.float32)
        else:
            image,mask,stem = load_raw(name)
        if self.augment:
            if random.random() < 0.5:
                image,mask = np.flip(image,1).copy(),np.flip(mask,1).copy()
            if random.random() < 0.5:
                image,mask = np.flip(image,0).copy(),np.flip(mask,0).copy()
            if random.random() < 0.75:
                gain = np.random.uniform(0.88,1.12,(1,1,3)).astype(np.float32)
                bias = np.random.uniform(-0.035,0.035,(1,1,3)).astype(np.float32)
                image[...,:3] = np.clip(image[...,:3]*gain+bias,0,1)
        image = ((image-MEAN)/STD).astype(np.float32)
        return torch.from_numpy(image.transpose(2,0,1)).float(),torch.from_numpy(mask[None]).float(),stem

def capture_rng(generator):
    return {'python':random.getstate(),'numpy':np.random.get_state(),
            'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if device.type=='cuda' else [],
            'loader':generator.get_state()}

def restore_rng(state,generator):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if device.type=='cuda' and state['cuda']:
        torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])
    generator.set_state(state['loader'].cpu())

def train_epoch(model,loader,optimizer,scaler,epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    rgb_weight,spectral_weight = auxiliary_loss_weights(epoch)
    total,main_total = 0.0,0.0
    for step,(images,masks,_) in enumerate(loader,start=1):
        images,masks = images.to(device,non_blocking=True),masks.to(device,non_blocking=True)
        # Correctly average the final partial accumulation window as well.
        window_start = ((step-1)//ACCUMULATION_STEPS)*ACCUMULATION_STEPS
        window_size = min(ACCUMULATION_STEPS,len(loader)-window_start)
        with torch.amp.autocast('cuda',enabled=device.type=='cuda'):
            main,rgb_aux,spec_aux = model(images,return_aux=True)
            main_loss = criterion(main,masks)
            loss = main_loss + rgb_weight*criterion(rgb_aux,masks) + spectral_weight*criterion(spec_aux,masks)
        if not torch.isfinite(loss).item():
            raise FloatingPointError(f'Non-finite loss at epoch {epoch}, batch {step}.')
        scaler.scale(loss/window_size).backward()
        total += float(loss.item())
        main_total += float(main_loss.item())
        if step % ACCUMULATION_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
    return {'loss':total/len(loader),'main_loss':main_total/len(loader),
            'rgb_aux_weight':rgb_weight,'spectral_aux_weight':spectral_weight}

@torch.no_grad()
def validate(model,loader):
    model.eval()
    totals = {t:np.zeros(4,dtype=np.int64) for t in TRAIN_THRESHOLDS}
    loss_total = 0.0
    for images,masks,_ in loader:
        images,masks = images.to(device),masks.to(device)
        with torch.amp.autocast('cuda',enabled=device.type=='cuda'):
            logits = model(images)
            loss = criterion(logits,masks)
        probabilities = torch.sigmoid(logits)
        if not torch.isfinite(probabilities).all().item():
            raise FloatingPointError('Non-finite validation predictions.')
        loss_total += float(loss.item())
        for threshold in TRAIN_THRESHOLDS:
            totals[threshold] += np.asarray(confusion(probabilities,masks,threshold),dtype=np.int64)
    rows = [{'threshold':float(t),'loss':loss_total/len(loader),**metrics(v.tolist())}
            for t,v in totals.items()]
    return max(rows,key=lambda r:r['dice'])

def threshold_confusions(probability,mask,thresholds):
    # Histogram method is exactly equivalent to probability >= threshold.
    # Float32 thresholds match comparisons made against float32 model outputs.
    threshold_array = np.asarray(thresholds,dtype=np.float32)
    foreground = np.asarray(mask)>=0.5
    pos = np.asarray(probability,dtype=np.float32)[foreground]
    neg = np.asarray(probability,dtype=np.float32)[~foreground]
    pos_hist = np.bincount(np.searchsorted(threshold_array,pos,side='right'),minlength=len(thresholds)+1)
    neg_hist = np.bincount(np.searchsorted(threshold_array,neg,side='right'),minlength=len(thresholds)+1)
    tp = np.cumsum(pos_hist[::-1],dtype=np.int64)[::-1][1:]
    fp = np.cumsum(neg_hist[::-1],dtype=np.int64)[::-1][1:]
    return np.column_stack([tp,fp,len(pos)-tp,len(neg)-fp])

@torch.no_grad()
def fine_validation(model,loader):
    model.eval()
    total = np.zeros((len(FINAL_THRESHOLDS),4),dtype=np.int64)
    for images,masks,_ in loader:
        # Original rescan cell uses float32, single-pass final inference.
        probability = torch.sigmoid(model(images.to(device)))[0,0].cpu().numpy()
        if not np.isfinite(probability).all():
            raise FloatingPointError('Non-finite predictions during fine validation.')
        total += threshold_confusions(probability,masks[0,0].numpy(),FINAL_THRESHOLDS)
    rows = [{'threshold':float(t),**metrics(v.tolist())} for t,v in zip(FINAL_THRESHOLDS,total)]
    return max(rows,key=lambda r:r['dice']),rows

@torch.no_grad()
def evaluate_test(model,loader,threshold,out):
    model.eval()
    totals = np.zeros(4,dtype=np.int64)
    rows = []
    if SAVE_TEST_PREDICTIONS:
        (out/'test_masks').mkdir(exist_ok=True)
        (out/'test_probabilities_uint16').mkdir(exist_ok=True)
    for images,masks,stems in tqdm(loader,desc='Test once / fixed validation threshold'):
        probability = torch.sigmoid(model(images.to(device)))[0,0].cpu().numpy()
        if not np.isfinite(probability).all():
            raise FloatingPointError('Non-finite test predictions.')
        values = confusion(probability,masks[0,0].numpy(),threshold)
        totals += np.asarray(values,dtype=np.int64)
        rows.append({'sample_name':stems[0],'threshold':threshold,**metrics(values),
                     **dict(zip(('tp','fp','fn','tn'),values))})
        if SAVE_TEST_PREDICTIONS:
            Image.fromarray(((probability>=threshold)*255).astype(np.uint8)).save(out/'test_masks'/f'{stems[0]}.png')
            Image.fromarray(np.rint(probability*65535).astype(np.uint16)).save(
                out/'test_probabilities_uint16'/f'{stems[0]}.png')
    save_csv(out/'test_per_image_metrics.csv',rows)
    return metrics(totals.tolist()),dict(zip(('tp','fp','fn','tn'),map(int,totals)))

def run_seed(seed):
    out = ROOT_OUT/f'seed{seed}'
    out.mkdir(exist_ok=True)
    result_path,last_path,best_path = out/'test_metrics.json',out/'last_checkpoint.pt',out/'best_checkpoint.pt'
    manifest_path = out/'run_config.json'
    run_id = hashlib.sha256((CONFIG_ID+f':seed{seed}').encode()).hexdigest()
    if manifest_path.exists():
        old = json.loads(manifest_path.read_text())
        if old.get('run_id') != run_id:
            raise RuntimeError(f'Existing output belongs to a different configuration: {out}. Set a new OUTPUT_DIR.')
        if not RESUME:
            raise RuntimeError(f'Output already exists: {out}. Set a new OUTPUT_DIR for a fresh rerun.')
    else:
        if any(out.iterdir()):
            raise RuntimeError(f'Unrecognized files in {out}; choose another OUTPUT_DIR.')
        save_json(manifest_path,{'run_id':run_id,'seed':seed,'protocol':PROTOCOL})
    if result_path.exists():
        result = json.loads(result_path.read_text())
        if result.get('run_id') != run_id:
            raise RuntimeError('Completed-result configuration mismatch.')
        print(f'Seed {seed}: already completed; reusing saved metrics.')
        return result
    seed_everything(seed)
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(BufferedDataset(TRAIN_NAMES,True),batch_size=BATCH_SIZE,
                              shuffle=True,num_workers=0,pin_memory=pin_memory,drop_last=False)
    val_loader = DataLoader(BufferedDataset(VAL_NAMES),batch_size=1,shuffle=False,
                            num_workers=0,pin_memory=pin_memory)
    model,params,aux_params,deploy_params = build_model()
    optimizer = torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=0.5,patience=4,min_lr=1e-6)
    scaler = torch.amp.GradScaler('cuda',enabled=device.type=='cuda')
    best_dice,best_epoch,history,start_epoch = -1.0,0,[],1
    if last_path.exists():
        checkpoint = load_checkpoint(last_path)
        if checkpoint['run_id'] != run_id:
            raise RuntimeError('Resume checkpoint configuration mismatch.')
        model.load_state_dict(checkpoint['model'],strict=True)
        optimizer.load_state_dict(checkpoint['optimizer'])
        scheduler.load_state_dict(checkpoint['scheduler'])
        scaler.load_state_dict(checkpoint['scaler'])
        best_dice,best_epoch = checkpoint['best_dice'],checkpoint['best_epoch']
        history = checkpoint['history']
        start_epoch = checkpoint['epoch']+1
        restore_rng(checkpoint['rng'],generator)
        print(f'Seed {seed}: resume from epoch {start_epoch}; best epoch {best_epoch}.')
        del checkpoint
    elif best_path.exists():
        raise RuntimeError('A best checkpoint exists without a last checkpoint. Choose a fresh OUTPUT_DIR.')
    print('\n'+'='*96)
    print(f'24 m BUFFERED | seed {seed} | TRAIN 207 / VALIDATION 40 / TEST 41 | NO TTA')
    print(f'Parameters: {params:,} training; {deploy_params:,} deployment; auxiliary {aux_params:,}')
    print('='*96)
    already_stopped = bool(history and history[-1]['epoch']-best_epoch >= PATIENCE)
    for epoch in range(start_epoch,MAX_EPOCHS+1):
        if already_stopped:
            break
        started = time.time()
        lr_used = float(optimizer.param_groups[0]['lr'])
        training = train_epoch(model,train_loader,optimizer,scaler,epoch)
        validation = validate(model,val_loader)
        scheduler.step(validation['dice'])
        row = {'epoch':epoch,'lr_used':lr_used,'lr_next':float(optimizer.param_groups[0]['lr']),
               'train_loss':training['loss'],'train_main_loss':training['main_loss'],
               'rgb_aux_weight':training['rgb_aux_weight'],'spectral_aux_weight':training['spectral_aux_weight'],
               'val_loss':validation['loss'],'val_threshold_coarse':validation['threshold'],
               **{f'val_{m}':validation[m] for m in METRIC_NAMES},
               'minutes':(time.time()-started)/60}
        history.append(row)
        if validation['dice'] > best_dice:
            best_dice,best_epoch = float(validation['dice']),epoch
            atomic_torch_save({'run_id':run_id,'seed':seed,'model':model.state_dict(),'epoch':epoch,
                               'val_dice_coarse':best_dice,'mean':MEAN.tolist(),'std':STD.tolist(),
                               'protocol':PROTOCOL},best_path)
        atomic_torch_save({'run_id':run_id,'seed':seed,'epoch':epoch,'model':model.state_dict(),
                           'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                           'scaler':scaler.state_dict(),'best_epoch':best_epoch,'best_dice':best_dice,
                           'history':history,'rng':capture_rng(generator)},last_path)
        save_csv(out/'training_history.csv',history)
        print(f"seed {seed} | epoch {epoch:03d} | loss {training['loss']:.4f} | "
              f"val Dice {validation['dice']:.4f} | IoU {validation['iou']:.4f} | "
              f"P {validation['precision']:.4f} | R {validation['recall']:.4f} | "
              f"th {validation['threshold']:.2f} | lr {lr_used:.2e}",flush=True)
        if epoch-best_epoch >= PATIENCE:
            print(f'Early stopping: {PATIENCE} non-improving epochs.')
            break
    checkpoint = load_checkpoint(best_path)
    model.load_state_dict(checkpoint['model'],strict=True)
    selected,threshold_rows = fine_validation(model,val_loader)
    threshold = float(selected['threshold'])
    save_csv(out/'validation_threshold_search.csv',threshold_rows)
    checkpoint.update({'selected_threshold':threshold,'validation_metrics_fine':selected,'no_tta':True})
    atomic_torch_save(checkpoint,best_path)
    del checkpoint
    # Test loader is created only after all checkpoint and threshold decisions.
    test_loader = DataLoader(BufferedDataset(TEST_NAMES),batch_size=1,shuffle=False,
                             num_workers=0,pin_memory=pin_memory)
    test_metrics,counts = evaluate_test(model,test_loader,threshold,out)
    result = {'run_id':run_id,'seed':seed,'best_epoch':int(best_epoch),
              'best_validation_dice_coarse':best_dice,'validation_dice_fine':selected['dice'],
              'selected_threshold':threshold,'train_images':207,'validation_images':40,'test_images':41,
              'buffer_m':24.0,'no_tta':True,'metric_aggregation':'pooled pixel confusion counts',
              **test_metrics,**counts,'output_dir':str(out)}
    save_json(result_path,result)
    save_csv(out/'test_metrics.csv',[result])
    print(f"\nFINAL seed {seed} | epoch {best_epoch} | th {threshold:.2f} | "
          f"Dice {test_metrics['dice']:.6f} | IoU {test_metrics['iou']:.6f} | "
          f"Precision {test_metrics['precision']:.6f} | Recall {test_metrics['recall']:.6f}")
    print('Saved:',out)
    del model,optimizer,scheduler,scaler,train_loader,val_loader,test_loader
    gc.collect()
    if device.type=='cuda':
        torch.cuda.empty_cache()
    return result

def summarize(results):
    fields = ('seed','best_epoch','selected_threshold','dice','iou','precision','recall','accuracy')
    save_csv(ROOT_OUT/'three_seed_test_results.csv',[{k:r[k] for k in fields} for r in results])
    rows = []
    for metric in METRIC_NAMES:
        values = np.asarray([r[metric] for r in results],dtype=np.float64)
        rows.append({'metric':metric,'n_seeds':len(results),'seeds':','.join(str(r['seed']) for r in results),
                     'mean':float(values.mean()),'sample_sd':float(values.std(ddof=1)) if len(values)>1 else None})
    if len(results)>1:
        save_csv(ROOT_OUT/'mean_sample_sd.csv',rows)
    save_json(ROOT_OUT/'summary.json',{'run_seeds':[r['seed'] for r in results],
                                      'results':results,'mean_sample_sd':rows})
    print('\nCurrent results:')
    for r in results:
        print(f"seed {r['seed']} | D {r['dice']:.6f} | I {r['iou']:.6f} | "
              f"P {r['precision']:.6f} | R {r['recall']:.6f}")
    if len(results)==len(RUN_SEEDS):
        print('\nFINAL THREE-SEED MEAN +/- SAMPLE SD (ddof=1):')
        for r in rows:
            print(f"{r['metric']:10s}: {r['mean']:.4f} +/- {r['sample_sd']:.4f}")

def package_results():
    # Lightweight archive: CSV/JSON and split lists. Full checkpoints remain in output folder.
    path = ROOT_OUT/'buffered24m_3seed_tables_and_protocol.zip'
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for item in sorted(ROOT_OUT.rglob('*')):
            if item.is_file() and item.suffix in ('.csv','.json','.txt'):
                archive.write(item,item.relative_to(ROOT_OUT))
    print('\nTables/protocol ZIP:',path)
    try:
        from IPython.display import display,FileLink
        display(FileLink(str(path)))
    except ImportError:
        pass

def main():
    global ROOT_OUT,rgb_dir,ms_dir,mark_dir,device,pin_memory
    global TRAIN_NAMES,VAL_NAMES,TEST_NAMES,RAM_CACHE,MEAN,STD,PROTOCOL,CONFIG_ID,criterion
    dataset,split,splits,excluded,audit,ROOT_OUT = initialize_inputs()
    rgb_dir,ms_dir,mark_dir = dataset/'RGB',dataset/'MS',dataset/'Mark'
    TRAIN_NAMES,VAL_NAMES,TEST_NAMES = splits['train'],splits['val'],splits['test']
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pin_memory = device.type=='cuda'
    if device.type != 'cuda':
        raise RuntimeError('Enable a GPU runtime before running the full 896x672 training experiment.')
    if len(set(RUN_SEEDS)) != len(RUN_SEEDS) or len(RUN_SEEDS)<2:
        raise ValueError('Use unique seeds; this runner expects multiple seeds.')
    print('='*96)
    print('GD | FIXED 24 m BUFFERED SPLIT | FINAL SIX-CHANNEL AUX-GATED MODEL')
    print('Dataset:',dataset,'\nSplit folder:',split,'\nOutput:',ROOT_OUT)
    print('Seeds:',RUN_SEEDS,'| Split: 207 / 40 / 41 | Input:',TARGET_SIZE,'| No TTA')
    print('Excluded images:',len(excluded))
    for row in audit:
        print(row['comparison'],f"minimum {row['minimum_camera_centre_distance_m']:.3f} m")
    print('Fine validation threshold grid: 0.20..0.95, step 0.01')
    RAM_CACHE,MEAN,STD = prepare_data(TRAIN_NAMES,VAL_NAMES)
    PROTOCOL = {'version':'buffered24m_auxfusion_verified_notebook_v2','dataset_root':str(dataset.resolve()),
                'model':'six-channel spatially gated fusion with two training-only auxiliary heads',
                'training_parameters':7950483,'deployment_parameters':7950033,
                'input_channel_order':list(INPUT_CHANNEL_ORDER),'target_size':list(TARGET_SIZE),
                'buffer_m':BUFFER_M,'distance_audit':'GPS haversine camera-centre distance',
                'split_lists':splits,'excluded_images':len(excluded),'normalization_source':'buffered training images only',
                'mean':MEAN.tolist(),'std':STD.tolist(),'max_epochs':MAX_EPOCHS,'patience':PATIENCE,
                'batch_size':BATCH_SIZE,'accumulation_steps':ACCUMULATION_STEPS,
                'lr':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,'scheduler':{'factor':0.5,'patience':4,'min_lr':1e-6},
                'main_loss':{'bce':0.35,'focal':0.25,'dice':0.40,'focal_gamma':2.0},
                'auxiliary_weights':{'initial':[0.15,0.25],'middle':[0.10,0.15],'final':[0.05,0.08],
                                     'phase_starts':[0.40,0.70]},
                'train_thresholds':TRAIN_THRESHOLDS,'final_thresholds':FINAL_THRESHOLDS,
                'no_tta':True,'cache_float16':CACHE_TRAIN_AND_VALIDATION,
                'loss_implementation':'copied unchanged from original 24 m training cell',
                'final_inference_precision':'float32, as in the original threshold rescan',
                'partial_accumulation_window_corrected':True,
                'metric_aggregation':'pooled pixel confusion counts','sd_ddof':1,
                'torch_version':str(torch.__version__)}
    CONFIG_ID = hashlib.sha256(json.dumps(PROTOCOL,sort_keys=True).encode()).hexdigest()
    protocol_path = ROOT_OUT/'protocol.json'
    if protocol_path.exists() and json.loads(protocol_path.read_text()).get('config_id') != CONFIG_ID:
        raise RuntimeError('Output folder contains a different protocol. Set a fresh OUTPUT_DIR.')
    for label,names in {**splits,'excluded_buffer':excluded}.items():
        (ROOT_OUT/f'{label}_list.txt').write_text('\n'.join(names)+'\n')
    if (split/'spatial_24m_split_assignments.csv').exists():
        shutil.copy2(split/'spatial_24m_split_assignments.csv',ROOT_OUT/'spatial_24m_split_assignments.csv')
    save_csv(ROOT_OUT/'cross_split_distance_audit.csv',audit)
    save_json(protocol_path,{'config_id':CONFIG_ID,**PROTOCOL})
    save_json(ROOT_OUT/'normalization_statistics.json',{'source':'207 buffered training images only',
              'n_images':len(TRAIN_NAMES),'channels':list(INPUT_CHANNEL_ORDER),'mean':MEAN.tolist(),'std':STD.tolist()})
    criterion = ComboLoss()
    results = []
    for seed in RUN_SEEDS:
        results.append(run_seed(seed))
        summarize(results)
    package_results()
    print('\nCompleted all three seeds. Full checkpoints and predictions:',ROOT_OUT)

if __name__ == '__main__':
    main()
